In [2]:
# Project: BPO Operations Analytics - Silver layer
# Description: Clean and NULL handle in Genesys data

from pyspark.sql import functions as F
from pyspark.sql.types import *


# 1. Origin path (bronze) and destiny (silver)
path_bronze_genesys = "Files/bronze/genesys/calls_log.csv"
path_silver_genesys = "Tables/silver_genesys" # It will be saved now under tables and not files


# 2. Bronze layer read
# Spark will read all of the files part-...

print("Reading bronze layer...")
df_bronze = spark.read.option("header", "true").csv(path_bronze_genesys)


# 3. Cleaning and transformation
print("Cleaning...")
df_silver = df_bronze.select(
    F.col("interaction_id").cast(StringType()).alias("interaction_id"),
    F.col("agent_id").cast(StringType()).alias("agent_id"),
    F.col("timestamp").cast(TimestampType()).alias("timestamp"),
    # Usamos alias() para que el nombre de la columna sea simple
    F.coalesce(F.col("talk_time_sec").cast(IntegerType()), F.lit(0)).alias("talk_time_sec"),
    F.col("hold_time_sec").cast(IntegerType()).alias("hold_time_sec"),
    F.col("acw_time_sec").cast(IntegerType()).alias("acw_time_sec"),
    F.col("queue_name").cast(StringType()).alias("queue_name")
)

# 4. AHT calculation
    # AHT = Talk + Hold + ACW
df_silver = df_silver.withColumn(
    "total_handle_time_sec",
    F.col("talk_time_sec") + F.col("hold_time_sec") + F.col("acw_time_sec")
)

# 5. Save as DELTA to create a real table to consult
print("Saving as a DELTA table in path_silver_genesys...")
df_silver.write.format("delta").mode("overwrite").saveAsTable("silver_genesys")
print("Behold, n\ Another offering!")

StatementMeta(, 919bd38c-4b62-4276-96a8-6008c02fda90, 4, Finished, Available, Finished)

Reading bronze layer...
Cleaning...
Saving as a DELTA table in path_silver_genesys...
Behold, n\ Another offering!
